In [32]:
#WAIT actually im going to use the cosmetic expansion from yesterday to modify instead of starting from scratch 
#deleting all the unnecessary stuff - no more functional group or alkene preservation filters --> these things can be used in post reaction processing if needed


import doranet as dn
from doranet.modules.synthetic.Reaction_Smarts_Forward import op_smarts
from doranet.modules.synthetic.Reaction_Smarts_Retro import op_retro_smarts
from doranet.modules.synthetic.generate_network import (
    get_smiles_from_file,
    Max_Atoms_Filter,
    Ring_Issues_Filter,
    Enol_filter_forward,
    Enol_filter_retro,
    Check_balance_filter,
    Allowed_Elements_Filter,
    Chem_Rxn_dH_Calculator,
    Rxn_dH_Filter,
    Cross_Reaction_Filter,
    Retro_Not_Aromatic_Filter,
)
from datetime import datetime
import time
from rdkit import Chem
from rdkit.Chem import Descriptors
from doranet import metadata, interfaces
import collections.abc
from tal_reaction_whitelist import TAL_REACTION_WHITELIST #THIS IS USED FOR CUSTOM WHITELIST


# ============ CUSTOM FILTERS ============

class Molecular_Weight_Filter(metadata.ReactionFilterBase):
    """Reject reactions if products exceed max molecular weight."""
    
    def __init__(self, max_weight=400):
        self.max_weight = max_weight
    
    def __call__(self, recipe):
        for mol in recipe.products:
            if not isinstance(mol.item, interfaces.MolDatRDKit):
                raise NotImplementedError(
                    f"Filter only works with MolDatRDKit, not {type(mol.item)}"
                )
            
            mw = Descriptors.MolWt(mol.item.rdkitmol)
            
            if mw > self.max_weight:
                return False
        
        return True
    
    @property
    def meta_required(self):
        return interfaces.MetaKeyPacket()


class Minimum_Carbon_Count_Filter(metadata.ReactionFilterBase):
    """Reject molecules with fewer than 8 carbons."""
    
    def __init__(self, min_carbons=0):
        self.min_carbons = min_carbons
    
    def __call__(self, recipe):
        for mol in recipe.products:
            if not isinstance(mol.item, interfaces.MolDatRDKit):
                raise NotImplementedError(
                    f"Filter only works with MolDatRDKit, not {type(mol.item)}"
                )
            
            # Count carbons
            carbon_count = sum(
                1 for atom in mol.item.rdkitmol.GetAtoms() 
                if atom.GetAtomicNum() == 6
            )
            
            if carbon_count < self.min_carbons:
                return False
        
        return True
    
    @property
    def meta_required(self):
        return interfaces.MetaKeyPacket()


# ============ CUSTOM GENERATE FUNCTION ============

def generate_network_tal(
    job_name="default_job",
    starters=False,
    helpers=False,
    gen=2, #very suspicious line i thought i was passing this in as a parameter -- ask ai wtf this line is doing
    direction="forward",
    molecule_thermo_calculator=None,
    max_rxn_thermo_change=15,
    max_atoms=None,  # {"C": 50, "O": 8, "N": 2}
    max_molecular_weight=800,  # NEW: Adjustable MW cap
    allow_multiple_reactants="default",
    targets=None,
    strategy="cartesian",  # NEW: "cartesian" or "priority_queue"
    #preserve_delta9=True,  # NEW: Preserve Δ9 alkene
    min_carbons=0,  # NEW: Minimum carbon count
    #enforce_functional_groups=True,  # NEW: Whitelist functional groups
):
    """
    Enhanced generate_network for TAL acid derivatives.
    
    Parameters:
    -----------
    max_atoms : dict
        Atom count limits, e.g., {"C": 50, "O": 8, "N": 2}
    
    max_molecular_weight : float
        Max molecular weight in Daltons (default: 800)
    
    strategy : str
        "cartesian" (blind, exhaustive) or "priority_queue" (targeted)
        If "priority_queue", auto-detects target from targets parameter
    
    min_carbons : int
        Minimum carbon count in molecules (default: 8)
    
    """
    
    if not starters:
        raise Exception("At least one starter is needed to generate a network")

    starters = get_smiles_from_file(starters)
    helpers = get_smiles_from_file(helpers)
    targets = get_smiles_from_file(targets)

    print(f"\n{'='*60}")
    print(f"TAL NETWORK GENERATION")
    print(f"{'='*60}")
    print(f"Job name: {job_name}")
    print(f"Job type: synthetic network expansion {direction}")
    print(f"Strategy: {strategy}")
    if strategy == "priority_queue" and targets:
        print(f"Priority queue targets: {targets}")
    print(f"Atom limits: {max_atoms}")
    print(f"Max molecular weight: {max_molecular_weight} Da")
    #print(f"Preserve Δ9 alkene: {preserve_delta9}")
    print(f"Min carbons: {min_carbons}")
    #print(f"Enforce functional groups: {enforce_functional_groups}")
    print("Job started on:", datetime.now())
    start_time = time.time()

    engine = dn.create_engine()
    network = engine.new_network()

    if helpers:
        for smiles in helpers:
            network.add_mol(engine.mol.rdkit(smiles))

    my_start_i = -1
    for smiles in starters:
        if my_start_i == -1:
            my_start_i = network.add_mol(engine.mol.rdkit(smiles))
        else:
            network.add_mol(engine.mol.rdkit(smiles))

    if direction == "forward":
        smarts_list = [op for op in op_smarts if op.name in TAL_REACTION_WHITELIST]
        #print("SMARTS LIST")
        #for smarts in smarts_list:
           # print(smarts.name)
            
        #TS IS NEW 
    elif direction == "retro":
        smarts_list = op_retro_smarts

    for smarts in smarts_list:
        if smarts.kekulize_flag is False:
            network.add_op(
                engine.op.rdkit(smarts.smarts, drop_errors=True),
                meta={
                    "name": smarts.name,
                    "reactants_stoi": smarts.reactants_stoi,
                    "products_stoi": smarts.products_stoi,
                    "enthalpy_correction": smarts.enthalpy_correction,
                    "ring_issue": smarts.ring_issue,
                    "kekulize_flag": smarts.kekulize_flag,
                    "Retro_Not_Aromatic": smarts.Retro_Not_Aromatic,
                    "number_of_steps": smarts.number_of_steps,
                    "allowed_elements": smarts.allowed_elements,
                    "Reaction_type": smarts.reaction_type,
                    "Reaction_direction": direction,
                },
            )
        if smarts.kekulize_flag is True:
            network.add_op(
                engine.op.rdkit(smarts.smarts, kekulize=True, drop_errors=True),
                meta={
                    "name": smarts.name,
                    "reactants_stoi": smarts.reactants_stoi,
                    "products_stoi": smarts.products_stoi,
                    "enthalpy_correction": smarts.enthalpy_correction,
                    "ring_issue": smarts.ring_issue,
                    "kekulize_flag": smarts.kekulize_flag,
                    "Retro_Not_Aromatic": smarts.Retro_Not_Aromatic,
                    "number_of_steps": smarts.number_of_steps,
                    "allowed_elements": smarts.allowed_elements,
                    "Reaction_type": smarts.reaction_type,
                    "Reaction_direction": direction,
                },
            )

    print(f"Number of operators added to network: {len(network.ops)}")

    # ====== CHOOSE STRATEGY ======
    if strategy == "cartesian":
        strat = engine.strat.cartesian(network)
    elif strategy == "priority_queue":
        if not targets:
            raise ValueError(
                "priority_queue strategy requires targets parameter to be set"
            )
        # Use first target as the ranker target
        target_for_ranker = targets[0] if isinstance(targets, list) else targets
        ranker = engine.ranker.smiles_distance(target_for_ranker)
        strat = engine.strat.priority_queue(network, ranker=ranker)
        print(f"Using priority queue strategy, targeting: {target_for_ranker}")
    else:
        raise ValueError(
            f"Unknown strategy: {strategy}. Use 'cartesian' or 'priority_queue'"
        )
    # =============================

    periodic_table = Chem.GetPeriodicTable()

    if max_atoms is None:
        max_atoms_dict_num = None
    else:
        max_atoms_dict_num = dict()
        for key in max_atoms:
            max_atoms_dict_num[periodic_table.GetAtomicNumber(key)] = max_atoms[key]

    # Build reaction plan with custom filters
    if direction == "forward":
        reaction_plan = (
            Max_Atoms_Filter(max_atoms_dict_num)
            >> Molecular_Weight_Filter(max_molecular_weight)
            >> Ring_Issues_Filter()
            >> Enol_filter_forward()
            >> Check_balance_filter()
            >> Allowed_Elements_Filter()
        )
        
        # Add optional filters
       #if preserve_delta9:
          #  reaction_plan = reaction_plan >> Preserve_Delta9_Alkene_Filter()
        
        if min_carbons > 0:
            reaction_plan = reaction_plan >> Minimum_Carbon_Count_Filter(min_carbons)
        
        #  enforce_functional_groups:
            #reaction_plan = reaction_plan >> Functional_Group_Whitelist_Filter()
        
        # Add thermodynamics filters at the end
        reaction_plan = (
            reaction_plan
            >> Chem_Rxn_dH_Calculator("dH", "forward", molecule_thermo_calculator)
            >> Rxn_dH_Filter(max_rxn_thermo_change, "dH")
        )
        #HOLD UP - is this from original or is this something i added - I DO NOT REMEMBER ADDING THIS
        #Should thermodynamics be filter or analysis --> actually im 80% sure this is from original so i wont touch
        
        recipe_filter = None

    elif direction == "retro":
        reaction_plan = (
            Max_Atoms_Filter(max_atoms_dict_num)
            >> Molecular_Weight_Filter(max_molecular_weight)
            >> Ring_Issues_Filter()
            >> Retro_Not_Aromatic_Filter()
            >> Enol_filter_retro()
            >> Allowed_Elements_Filter()
            >> Check_balance_filter()
        )
        
        # Add optional filters
        #if preserve_delta9:
         #   reaction_plan = reaction_plan >> Preserve_Delta9_Alkene_Filter()"""
        
        if min_carbons > 0:
            reaction_plan = reaction_plan >> Minimum_Carbon_Count_Filter(min_carbons)
        
        #if enforce_functional_groups:
         #   reaction_plan = reaction_plan >> Functional_Group_Whitelist_Filter()"""
        
        # Add thermodynamics filters at the end
        reaction_plan = (
            reaction_plan
            >> Chem_Rxn_dH_Calculator("dH", "retro", molecule_thermo_calculator)
            >> Rxn_dH_Filter(max_rxn_thermo_change, "dH")
        )
        
        recipe_filter = Cross_Reaction_Filter(tuple(range(my_start_i)))

    if allow_multiple_reactants != "default":
        if allow_multiple_reactants is True:
            recipe_filter = None
        elif allow_multiple_reactants is False:
            recipe_filter = Cross_Reaction_Filter(tuple(range(my_start_i)))

    bundle_filter = engine.filter.bundle.coreactants(tuple(range(my_start_i)))

    ini_number = len(network.mols)

    strat.expand(
        num_iter=gen,
        reaction_plan=reaction_plan,
        bundle_filter=bundle_filter,
        recipe_filter=recipe_filter,
        save_unreactive=False,
    )

    if targets is not None:
        print("\nChecking for targets...")
        to_check = set()
        if isinstance(targets, str):
            to_check.add(Chem.MolToSmiles(Chem.MolFromSmiles(targets)))
        else:
            for i in targets:
                to_check.add(Chem.MolToSmiles(Chem.MolFromSmiles(i)))

        targets_found = []
        for mol in network.mols:
            if (
                network.reactivity[network.mols.i(mol.uid)] is True
                and mol.uid in to_check
            ):
                targets_found.append(mol.uid)
                print(f"  ✓ Target found: {mol.uid}")
        
        if len(targets_found) == 0:
            print(f"  ✗ No targets found")

    print("\n" + "="*60)
    print("NETWORK GENERATION COMPLETE")
    print("="*60)
    print(f"Number of generations: {gen}")
    print(f"Number of operators: {len(network.ops)}")
    print(f"Number of molecules before expansion: {ini_number}")
    print(f"Number of molecules after expansion: {len(network.mols)}")
    print(f"Number of reactions: {len(network.rxns)}")

    end_time = time.time()
    elapsed_time = (end_time - start_time) / 60
    print(f"Time used: {elapsed_time:.2f} minutes")
    print("="*60 + "\n")

    network.save_to_file(f"{job_name}_{direction}_saved_network")

    return network

In [33]:

# Use cell
network = generate_network_tal(
  starters=["CC1=CC(O)=CC(=O)O1"],
    helpers=["[H][H]", "O", "CCO"],
    max_atoms={
        "C": 25,
        "O": 4,
        "N": 2,
        "S": 0,
        "P": 0,
        "F": 0,
        "Cl": 0,
        "Br": 0,
        "I": 0,
    },
    job_name="tal_basic",
    gen=3,
    direction="forward",
    min_carbons = 0,
    
)

# number of molecules and rxns 
print(f"Generated {len(network.mols)} molecules")
print(f"Generated {len(network.rxns)} reactions")

# Print all molecules
for i, mol in enumerate(network.mols):
    print(f"Molecule {i}: {mol.uid}")  # uid is the SMILES string


#WAIT MAJOR ISSUE - all the things i thought i was passing in to the generation network are actually being hardcoded in. 



TAL NETWORK GENERATION
Job name: tal_basic
Job type: synthetic network expansion forward
Strategy: cartesian
Atom limits: {'C': 25, 'O': 4, 'N': 2, 'S': 0, 'P': 0, 'F': 0, 'Cl': 0, 'Br': 0, 'I': 0}
Max molecular weight: 800 Da
Min carbons: 0
Job started on: 2026-06-10 03:19:37.633465
Number of operators added to network: 152


[03:19:37] reactant 1 has no mapped atoms.
[03:19:38] reactant 1 has no mapped atoms.
[03:19:39] reactant 1 has no mapped atoms.
[03:19:39] reactant 1 has no mapped atoms.
[03:19:39] reactant 1 has no mapped atoms.
[03:19:39] reactant 1 has no mapped atoms.
[03:19:39] reactant 2 has no mapped atoms.



NETWORK GENERATION COMPLETE
Number of generations: 3
Number of operators: 152
Number of molecules before expansion: 4
Number of molecules after expansion: 123
Number of reactions: 149
Time used: 0.11 minutes

Generated 123 molecules
Generated 149 reactions
Molecule 0: [H][H]
Molecule 1: O
Molecule 2: CCO
Molecule 3: Cc1cc(O)cc(=O)o1
Molecule 4: Cc1cccc(=O)o1
Molecule 5: CCOc1cc(C)oc(=O)c1
Molecule 6: CCc1c(O)cc(=O)oc1C
Molecule 7: CCc1c(O)cc(C)oc1=O
Molecule 8: CCc1cc(C)oc(=O)c1
Molecule 9: CCc1ccc(=O)oc1C
Molecule 10: CCc1ccc(C)oc1=O
Molecule 11: CC1=CC2C3C=CC(C)(OC3=O)C2C(=O)O1
Molecule 12: CC12C=CC(C(=O)O1)C1C=CC(=O)OC12C
Molecule 13: CC1=CC2C(C(=O)O1)C1C=CC2(C)OC1=O
Molecule 14: CC12C=CC(C(=O)O1)C1(C)OC(=O)C=CC21
Molecule 15: CC
Molecule 16: CCOc1cc(=O)oc(C)c1CC
Molecule 17: CCOc1cc(C)oc(=O)c1CC
Molecule 18: CCc1c(C)oc(=O)c(CC)c1O
Molecule 19: CCc1cc(=O)oc(C)c1CC
Molecule 20: CCc1cc(C)oc(=O)c1CC
Molecule 21: CCC12C=C(C)OC(=O)C1C1(C)C=CC2C(=O)O1
Molecule 22: CCC1=CC(=O)OC2(C)C1C1C=

In [48]:
from doranet.modules.post_processing.post_processing import (
    pretreat_networks,
    pathway_finder
)

STARTER = "Cc1cc(O)cc(=O)o1"

all_smiles = {mol.uid for mol in network.mols}

print(STARTER in all_smiles)


pretreat_networks(
    networks=[network],
    starters=[STARTER],
    total_generations=3,
    job_name="TAL"
)

target = "CCc1ccc(C)oc1=O"

print("Target exists:", target in {m.uid for m in network.mols})

pathway_finder(
    starters=[STARTER],
    target=[target],
    search_depth=3,
    max_num_rxns=20,
    job_name="TAL"
)

print ("pre pretreat # reactions")
print (len(network.rxns))
print ("post pretreat # reactions")
print (len(json.load(open("TAL_network_pretreated.json"))))

True
Job name: TAL
Job type: post-processing pretreat networks
Job started on: 2026-06-10 06:24:14.664837
Loading networks, it may take a while if loading large networks from file
Loading network 1 from memory
Number of reations in network 1: 149
Networks loaded, now generating reaction strings
Reaction strings generation finished
Removing unconnected reactions
Unconnected reactions removed
Total number of reactions after pretreatment: 0
Time used for network pretreatment: 0.00 minutes

Target exists: True
Job name: TAL
Job type: pathway search
Job started on: 2026-06-10 06:24:14.687634
Pathway finder started, total number of reactions in network 0
Searching for pathways.
If it is taking too long, try adjusting pruning parameters
No pathway found! Try adjusting pruning parameters.
Pathway search finished, removing loops if there's any.
Time used for pathway search: 0.00 minutes

second to last
149
last
0


In [31]:
#POST PROCESSING


#AI SLOP

print(f"Generated {len(network.mols)} molecules")
print(f"Generated {len(network.rxns)} reactions")

all_products = []
for i, mol in enumerate(network.mols):
    if network.reactivity[network.mols.i(mol.uid)] is True:
        smiles = mol.uid  # UID is the SMILES
        all_products.append({
            "index": i,
            "smiles": smiles,
            "uid": mol.uid,
        })

print(f"Total reactive products: {len(all_products)}")



#OK MY ACTUAL LOGIC ON HOW TO DO THIS

#First processing task - functional tags (like based on application)

# create a 'categories' results dictionary. create different boolean functions for each
# category of functional molecule (preservative, food, blah blah balh). iterate through 
# the list of molecules in the network and run EACH of the booleans. add to the respective\
# category based on what checks it passes. 

# pretty simple strategy i can't even lie. i think the main task will be to research what
# kind of checks go in each boolean function. 


categories = {
    "preservatives": [],
    "cosmetics": [],
    "ligands": [],
    "materials": [],
}


# CREATE EACH OF THOSE CATEGORIES RIGHT NOW. 

for i, mol in enumerate(network.mols):
    if network.reactivity[network.mols.i(mol.uid)] is True:
        smiles = mol.uid  # UID is the SMILES
            if is_preservative_candidate(props):
                categories["preservatives"].append(record)
            if is_cosmetic_candidate(props):
                categories["cosmetics"].append(record)       
            if is_ligand_candidate(props):
                categories["ligands"].append(record)      
            if is_material_candidate(props):
                categories["materials"].append(record)





Generated 123 molecules
Generated 149 reactions
Total reactive products: 123


In [ ]:
# =========================
# Cell 3: Post-network analysis scaffold
# =========================

from rdkit import Chem
from rdkit.Chem import Descriptors

# ---------- Placeholder category logic ----------
# Replace pseudocode comments with real logic later.

def is_antimicrobial_candidate(mol, props):
    """
    PSEUDOCODE:
    - Check if molecule has motifs similar to dehydroacetic acid / pogostone
    - Check for pyrone/lactone core
    - Check for keto side chain / oxygenated ring / conjugated carbonyl system
    - Return (True/False, reasons_list)
    """
    reasons = []
    
    # Example pseudocode only:
    # if has_pyrone_core:
    #     reasons.append("has pyrone core")
    # if has_keto_side_chain:
    #     reasons.append("has keto side chain")
    
    return False, reasons


def is_bioactive_non_antimicrobial_candidate(mol, props):
    """
    PSEUDOCODE:
    - Check for motifs associated with non-antimicrobial bioactivity
    - Example: styrenylpyrone-like scaffold, extended conjugation, aromatic substitution
    - Exclude things that are already clearly antimicrobial-first
    - Return (True/False, reasons_list)
    """
    reasons = []
    
    # Example pseudocode only:
    # if has_styrenyl_like_motif:
    #     reasons.append("has styrenyl-like motif")
    
    return False, reasons


def is_commodity_intermediate_candidate(mol, props):
    """
    PSEUDOCODE:
    - Check if molecule is a simple building block / intermediate
    - Example: small oxygenated molecule, acid, ester, unsaturated acid, diketone
    - Exclude highly specialized / obviously bioactive structures
    - Return (True/False, reasons_list)
    """
    reasons = []
    
    # Example pseudocode only:
    # if props["mw"] < 250:
    #     reasons.append("small molecule range")
    # if has_simple_acid_or_ester:
    #     reasons.append("simple intermediate functionality")
    
    return False, reasons


# ---------- Placeholder tag logic ----------
# Tags are smaller descriptive labels, not major buckets.

def is_food_related(mol, props):
    """
    PSEUDOCODE:
    - Check if molecule resembles preservative / food-compatible acid type chemistry
    - Example: sorbic-acid-like, dehydroacetic-acid-like, low MW, oxygenated acid
    """
    return False


def is_cosmetic_related(mol, props):
    """
    PSEUDOCODE:
    - Check if molecule fits likely cosmetic/personal care ingredient space
    - Example: moderate MW, oxygenated functionality, not too reactive, plausible topical utility
    """
    return False


# ---------- Property calculator ----------
# Compute reusable descriptors once per molecule.

def compute_basic_properties(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None, None
    
    props = {
        "smiles": smiles,
        "mw": Descriptors.MolWt(mol),
        "num_carbons": sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 6),
        "num_oxygens": sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 8),
        "num_nitrogens": sum(1 for atom in mol.GetAtoms() if atom.GetAtomicNum() == 7),
        "ring_count": mol.GetRingInfo().NumRings(),
    }
    
    return mol, props


# ---------- Main data structures ----------

# Major category buckets
categories = {
    "antimicrobial": [],
    "bioactive_non_antimicrobial": [],
    "commodity_intermediate": [],
}

# All molecule records go here
all_records = []


# ---------- Classification loop ----------

for i, mol_entry in enumerate(network.mols):
    smiles = mol_entry.uid   # In your setup, uid is the SMILES string
    
    mol, props = compute_basic_properties(smiles)
    if mol is None:
        continue
    
    record = {
        "index": i,
        "smiles": smiles,
        "properties": props,
        "categories": set(),   # major buckets
        "tags": set(),         # smaller labels like "food", "cosmetic"
        "reasons": {},         # e.g. {"antimicrobial": ["has pyrone core"]}
    }
    
    # ---- Major categories ----
    is_match, reasons = is_antimicrobial_candidate(mol, props)
    if is_match:
        record["categories"].add("antimicrobial")
        record["reasons"]["antimicrobial"] = reasons
        categories["antimicrobial"].append(record)
    
    is_match, reasons = is_bioactive_non_antimicrobial_candidate(mol, props)
    if is_match:
        record["categories"].add("bioactive_non_antimicrobial")
        record["reasons"]["bioactive_non_antimicrobial"] = reasons
        categories["bioactive_non_antimicrobial"].append(record)
    
    is_match, reasons = is_commodity_intermediate_candidate(mol, props)
    if is_match:
        record["categories"].add("commodity_intermediate")
        record["reasons"]["commodity_intermediate"] = reasons
        categories["commodity_intermediate"].append(record)
    
    # ---- Tags ----
    if is_food_related(mol, props):
        record["tags"].add("food")
    
    if is_cosmetic_related(mol, props):
        record["tags"].add("cosmetic")
    
    all_records.append(record)


# ---------- Quick summary ----------

print(f"Total molecule records processed: {len(all_records)}")
print(f"Antimicrobial candidates: {len(categories['antimicrobial'])}")
print(f"Bioactive non-antimicrobial candidates: {len(categories['bioactive_non_antimicrobial'])}")
print(f"Commodity intermediates: {len(categories['commodity_intermediate'])}")


# ---------- Optional preview ----------

for rec in all_records[:10]:
    print("-" * 60)
    print("SMILES:", rec["smiles"])
    print("Categories:", sorted(rec["categories"]))
    print("Tags:", sorted(rec["tags"]))
    print("MW:", rec["properties"]["mw"])